# CNN + BiLTSM

## Cargar librerías

In [1]:
import os, random, numpy as np
from pathlib import Path
from typing import Dict, Any

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor, Callback
from pytorch_lightning.loggers import CSVLogger
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score

## CONFIG

In [8]:
DATA_ROOT    = Path(r"C:\Users\IALAB\Documents\GitHub\data\segmentación5seg_LogMel2D\segmentación5seg_LogMel2D")   # <-- AJUSTA
SPLITS       = {"train": "train", "val": "val", "test": "test"}

IMG_SIZE     = (224, 224)
BATCH_SIZE   = 128
NUM_WORKERS  = 0 if os.name == "nt" else 4
MAX_EPOCHS   = 100
LR           = 2e-4
WD           = 1e-4
DROPOUT      = 0.3
LSTM_HIDDEN  = 256
LSTM_LAYERS  = 2
SEED         = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")

## DATAMODULE

In [9]:
class SpectrogramDataModule(pl.LightningDataModule):
    def __init__(self, data_root: Path, splits: dict, batch_size: int, num_workers: int):
        super().__init__()
        self.data_root   = Path(data_root)
        self.splits      = splits
        self.batch_size  = batch_size
        self.num_workers = num_workers
        self.tx_train = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize(IMG_SIZE, antialias=True),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5]),
        ])
        self.tx_eval = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize(IMG_SIZE, antialias=True),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5]),
        ])
    def setup(self, stage=None):
        root = self.data_root
        self.ds_train = ImageFolder(root / self.splits["train"], transform=self.tx_train)
        self.ds_val   = ImageFolder(root / self.splits["val"],   transform=self.tx_eval)
        self.ds_test  = ImageFolder(root / self.splits["test"],  transform=self.tx_eval)
        self.num_classes = len(self.ds_train.classes)
    def train_dataloader(self):
        return DataLoader(self.ds_train, batch_size=self.batch_size, shuffle=True,
                          num_workers=self.num_workers, pin_memory=True)
    def val_dataloader(self):
        return DataLoader(self.ds_val, batch_size=self.batch_size, shuffle=False,
                          num_workers=self.num_workers, pin_memory=True)
    def test_dataloader(self):
        return DataLoader(self.ds_test, batch_size=self.batch_size, shuffle=False,
                          num_workers=self.num_workers, pin_memory=True)

## MODELO

In [10]:
class CNNEncoder(nn.Module):
    def __init__(self, in_ch=1, dropout=0.3):
        super().__init__()
        c1,c2,c3,c4 = 32,64,128,256
        self.block1 = nn.Sequential(
            nn.Conv2d(in_ch,c1,3,padding=1), nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
            nn.Conv2d(c1,c1,3,padding=1),    nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
            nn.MaxPool2d((2,2)), nn.Dropout(dropout)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(c1,c2,3,padding=1), nn.BatchNorm2d(c2), nn.ReLU(inplace=True),
            nn.Conv2d(c2,c2,3,padding=1), nn.BatchNorm2d(c2), nn.ReLU(inplace=True),
            nn.MaxPool2d((2,2)), nn.Dropout(dropout)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(c2,c3,3,padding=1), nn.BatchNorm2d(c3), nn.ReLU(inplace=True),
            nn.Conv2d(c3,c3,3,padding=1), nn.BatchNorm2d(c3), nn.ReLU(inplace=True),
            nn.MaxPool2d((2,2)), nn.Dropout(dropout)
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(c3,c4,3,padding=1), nn.BatchNorm2d(c4), nn.ReLU(inplace=True),
            nn.Conv2d(c4,c4,3,padding=1), nn.BatchNorm2d(c4), nn.ReLU(inplace=True),
            nn.MaxPool2d((2,1)), nn.Dropout(dropout)   # pool solo en frecuencia
        )
    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.block4(x)
        return x  # [B,256,F',T']

class CNNBiLSTMClassifier(pl.LightningModule):
    def __init__(self, num_classes, lr=2e-4, wd=1e-4, dropout=0.3, lstm_hidden=256, lstm_layers=2):
        super().__init__()
        self.save_hyperparameters()
        self.enc = CNNEncoder(1, dropout)
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))           # [B,256,1,T']
        self.lstm = nn.LSTM(256, lstm_hidden, num_layers=lstm_layers,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(2*lstm_hidden, 2*lstm_hidden), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(2*lstm_hidden, num_classes)
        )
        self.crit = nn.CrossEntropyLoss(label_smoothing=0.1)
        # métricas
        self.train_acc  = MulticlassAccuracy(num_classes=num_classes, average='macro')
        self.val_acc    = MulticlassAccuracy(num_classes=num_classes, average='macro')
        self.test_acc   = MulticlassAccuracy(num_classes=num_classes, average='macro')
        self.train_prec = MulticlassPrecision(num_classes=num_classes, average='macro')
        self.val_prec   = MulticlassPrecision(num_classes=num_classes, average='macro')
        self.test_prec  = MulticlassPrecision(num_classes=num_classes, average='macro')
        self.train_rec  = MulticlassRecall(num_classes=num_classes, average='macro')
        self.val_rec    = MulticlassRecall(num_classes=num_classes, average='macro')
        self.test_rec   = MulticlassRecall(num_classes=num_classes, average='macro')
        self.train_f1   = MulticlassF1Score(num_classes=num_classes, average='macro')
        self.val_f1     = MulticlassF1Score(num_classes=num_classes, average='macro')
        self.test_f1    = MulticlassF1Score(num_classes=num_classes, average='macro')

    def forward(self, x):
        f = self.enc(x)                       # [B,256,F',T']
        f = self.freq_pool(f).squeeze(2)      # [B,256,T']
        f = f.permute(0,2,1)                  # [B,T',256]
        out,_ = self.lstm(f)                  # [B,T',2*h]
        h = out.mean(dim=1)                   # [B,2*h]
        return self.head(h)                   # [B,C]

    def _shared_step(self, batch, stage):
        x,y = batch
        logits = self(x)
        loss = self.crit(logits, y)
        preds = logits.argmax(1)
        if stage=='train':
            self.train_acc.update(preds,y); self.train_prec.update(preds,y)
            self.train_rec.update(preds,y); self.train_f1.update(preds,y)
            self.log('train/loss', loss, on_step=False, on_epoch=True, prog_bar=False)
        elif stage=='val':
            self.val_acc.update(preds,y); self.val_prec.update(preds,y)
            self.val_rec.update(preds,y); self.val_f1.update(preds,y)
            self.log('val/loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        else:
            self.test_acc.update(preds,y); self.test_prec.update(preds,y)
            self.test_rec.update(preds,y); self.test_f1.update(preds,y)
            self.log('test/loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):   return self._shared_step(batch,'train')
    def validation_step(self, batch, batch_idx): return self._shared_step(batch,'val')
    def test_step(self, batch, batch_idx):       return self._shared_step(batch,'test')

    def on_train_epoch_end(self):
        acc=self.train_acc.compute(); self.train_acc.reset()
        pr =self.train_prec.compute(); self.train_prec.reset()
        rc =self.train_rec.compute();  self.train_rec.reset()
        f1 =self.train_f1.compute();   self.train_f1.reset()
        self.log_dict({'train/acc':acc,'train/prec':pr,'train/rec':rc,'train/f1':f1}, prog_bar=True)

    def on_validation_epoch_end(self):
        acc=self.val_acc.compute(); self.val_acc.reset()
        pr =self.val_prec.compute(); self.val_prec.reset()
        rc =self.val_rec.compute();  self.val_rec.reset()
        f1 =self.val_f1.compute();   self.val_f1.reset()
        self.log_dict({'val/acc':acc,'val/prec':pr,'val/rec':rc,'val/f1':f1}, prog_bar=True)

    def on_test_epoch_end(self):
        acc=self.test_acc.compute(); self.test_acc.reset()
        pr =self.test_prec.compute(); self.test_prec.reset()
        rc =self.test_rec.compute();  self.test_rec.reset()
        f1 =self.test_f1.compute();   self.test_f1.reset()
        self.log_dict({'test/acc':acc,'test/prec':pr,'test/rec':rc,'test/f1':f1}, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=LR, weight_decay=WD)
        def sch_lambda(epoch):
            warmup=5
            if epoch < warmup: return (epoch+1)/warmup
            t = (epoch-warmup)/max(1,(MAX_EPOCHS-warmup))
            return 0.5*(1+np.cos(np.pi*t))
        sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=sch_lambda)
        return {"optimizer": opt, "lr_scheduler": sch}

## Callback

In [11]:
class EpochMetricsPrinter(Callback):
    """Imprime métricas de train/val al cierre de cada época."""
    KEYS = [
        'train/loss','train/acc','train/prec','train/rec','train/f1',
        'val/loss','val/acc','val/prec','val/rec','val/f1'
    ]
    def _fmt(self, x: Any):
        try:
            return f"{float(x):.4f}"
        except Exception:
            return str(x)
    def on_validation_epoch_end(self, trainer: pl.Trainer, pl_module: pl.LightningModule):
        m: Dict[str, Any] = trainer.callback_metrics
        e = trainer.current_epoch
        vals = {k:self._fmt(m.get(k, float('nan'))) for k in self.KEYS}
        print(
            f"[Epoch {e:03d}] "
            f"TRAIN: loss={vals['train/loss']}  acc={vals['train/acc']}  "
            f"prec={vals['train/prec']}  rec={vals['train/rec']}  f1={vals['train/f1']}  |  "
            f"VAL: loss={vals['val/loss']}  acc={vals['val/acc']}  "
            f"prec={vals['val/prec']}  rec={vals['val/rec']}  f1={vals['val/f1']}"
        )

## Entrenamiento

In [12]:
def run_training():
    dm = SpectrogramDataModule(DATA_ROOT, SPLITS, BATCH_SIZE, NUM_WORKERS)
    dm.setup()
    print("Clases:", dm.ds_train.classes)
    print(f"train={len(dm.ds_train)} | val={len(dm.ds_val)} | test={len(dm.ds_test)}")

    model = CNNBiLSTMClassifier(num_classes=dm.num_classes, dropout=DROPOUT,
                                lstm_hidden=LSTM_HIDDEN, lstm_layers=LSTM_LAYERS)

    ckpt  = ModelCheckpoint(monitor='val/acc', mode='max', save_top_k=1, save_last=True,
                            filename='cnnlstm-{epoch:02d}-{val_acc:.3f}')
    early = EarlyStopping(monitor='val/acc', mode='max', patience=10)
    lrmon = LearningRateMonitor(logging_interval='epoch')
    logger = CSVLogger("logs", name="cnnlstm")
    printer = EpochMetricsPrinter()

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator='auto', devices='auto',
        precision='16-mixed',
        callbacks=[ckpt, early, lrmon, printer],   # <--- imprime por época
        logger=[logger],
        log_every_n_steps=25
    )

    trainer.fit(model, datamodule=dm)
    print("Best ckpt:", ckpt.best_model_path or "<none>", "| best:", ckpt.best_model_score)
    trainer.test(model=None, datamodule=dm, ckpt_path=('best' if ckpt.best_model_path else None))
    print("CSV logs:", logger.log_dir)

if __name__ == "__main__":
    run_training()

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Clases: ['Chimayche', 'Cumbia andina', 'Diablada', 'Huaylash', 'Huayno', 'Muliza', 'Santiago', 'Tinku', 'Tunantada', 'Yaravi (Harawi)']
train=58730 | val=2390 | test=1199


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name       | Type                | Params | Mode 
------------------------------------------------------------
0  | enc        | CNNEncoder          | 1.2 M  | train
1  | freq_pool  | AdaptiveAvgPool2d   | 0      | train
2  | lstm       | LSTM                | 2.6 M  | train
3  | head       | Sequential          | 267 K  | train
4  | crit       | CrossEntropyLoss    | 0      | train
5  | train_acc  | MulticlassAccuracy  | 0      | train
6  | val_acc    | MulticlassAccuracy  | 0      | train
7  | test_acc   | MulticlassAccuracy  | 0      | train
8  | train_prec | MulticlassPrecision | 0      | train
9  | val_prec   | MulticlassPrecision | 0      | train
10 | test_prec  | MulticlassPrecision | 0      | train
11 | train_rec  | MulticlassRecall    | 0      | train
12 | val_rec    | MulticlassRecall    | 0      | train
13 | test_rec   | MulticlassRecall    | 0      | train
14 | train_f1   | MulticlassF1Score   | 0      | train
15 | val_f1     

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

[Epoch 000] TRAIN: loss=nan  acc=nan  prec=nan  rec=nan  f1=nan  |  VAL: loss=2.3320  acc=nan  prec=nan  rec=nan  f1=nan


...

Validation: |          | 0/? [00:00<?, ?it/s]

[Epoch 051] TRAIN: loss=0.5163  acc=0.9934  prec=0.9934  rec=0.9934  f1=0.9934  |  VAL: loss=1.1794  acc=0.7870  prec=0.7910  rec=0.7870  f1=0.7869
Best ckpt: logs\cnnlstm\version_0\checkpoints\cnnlstm-epoch=41-val_acc=0.000.ckpt | best: tensor(0.8074, device='cuda:0')


Restoring states from the checkpoint path at logs\cnnlstm\version_0\checkpoints\cnnlstm-epoch=41-val_acc=0.000.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at logs\cnnlstm\version_0\checkpoints\cnnlstm-epoch=41-val_acc=0.000.ckpt
c:\Users\IALAB\anaconda3\envs\TesisRichiFix\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test/acc            0.8079341650009155
         test/f1            0.8043105602264404
        test/loss            1.144914150238037
        test/prec           0.8132778406143188
        test/rec            0.8079341650009155
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
CSV logs: logs\cnnlstm\version_0
